# Lab 3: huấn luyện phát hiện ba trạng thái khẩu trang
Chọn **Runtime > Change runtime type > T4 GPU** (nếu tài khoản còn quota). Notebook chỉ dùng Colab để huấn luyện, API sẽ chạy trên Render.

Trước khi chạy: đọc HUONG_DAN_LAB3.md; rà soát data-review.html; chạy `python pack_data.py` trên Windows. Cần hai file `lab3-source.zip` và `data-yolo.zip`. Không chia lại test trong notebook.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

print("Chọn lab3-source.zip và data-yolo.zip")
uploaded = files.upload()
for name in ["lab3-source.zip", "data-yolo.zip"]:
    if not Path(name).is_file():
        raise FileNotFoundError(name)
    with zipfile.ZipFile(name) as archive:
        base = Path.cwd().resolve()
        for member in archive.infolist():
            if not (base / member.filename).resolve().is_relative_to(base):
                raise ValueError("Unsafe ZIP path")
        archive.extractall(base)
del uploaded
print("Dataset và mã nguồn đã sẵn sàng")

In [ ]:
%pip install -r requirements-train.txt


Nếu Colab báo cần restart sau cài đặt, chọn **Runtime > Restart session**, rồi chạy từ ô kiểm tra GPU dưới đây. File trên máy ảo vẫn còn. Không cần upload lần nữa.

Thiết lập là đề xuất ban đầu: YOLO11n pretrained, ảnh 416, 50 epochs, batch 8. Early stopping dựa trên validation. Test chỉ dùng đánh giá cuối. GPU miễn phí không được bảo đảm.

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Đổi runtime, hoặc chạy train.py --device cpu trên Windows.")


In [ ]:
!python train.py --data data/yolo/data.yaml --epochs 50 --imgsz 416 --batch 8 --device 0


Đọc kết quả từng lớp trong `evidence/test-metrics.json` và `evidence/onnx-test-metrics.json`. Kiểm tra `evidence/export-parity.json` (Pillow và OpenCV có thể chênh nhẹ ở biên confidence). Giữ biểu đồ huấn luyện, confusion matrix, ảnh dự đoán và metadata mô hình. Không ghi số liệu từ hướng dẫn thay cho kết quả chạy thực tế.

In [ ]:
import json
print(Path("evidence/model-info.json").read_text())
print(Path("evidence/onnx-test-metrics.json").read_text())


In [ ]:
import zipfile
with zipfile.ZipFile("lab3-trained-output.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for directory in ["assets", "evidence", "runs"]:
        for file in Path(directory).rglob("*"):
            if file.is_file():
                archive.write(file, str(file))
files.download("lab3-trained-output.zip")


Giải nén output về thư mục lab trên Windows. File `assets/model.onnx` là mô hình triển khai duy nhất. Không đưa toàn bộ PyTorch/Ultralytics lên Render. Chạy API local trước rồi triển khai theo hướng dẫn.